<div dir="rtl">

# پروژه‌ی ۱ — راه‌حل مرجع (فقط برای مدرس)

یک پیاده‌سازی از همه‌ی الزامات، برای مقایسه با گزارش‌ها و طرح سؤال در دفاع. تصمیم‌های پاک‌سازی (مثلاً رد کردن نمره‌ی <code dir="ltr">145</code> به جای اصلاح آن به <code dir="ltr">14.5</code>، یا نگه داشتن سطر اول از دو سطر متناقض) قابل بحث‌اند؛ دانشجو می‌تواند تصمیم دیگری بگیرد، به شرط آنکه آن را توضیح دهد. اعداد او در این صورت کمی متفاوت می‌شود.

### ایرادهای کاشته‌شده در داده

| ایراد | تعداد | سطر |
|---|---|---|
| سطر عنوان و سطرهای خالی | ۱ + ۳ | |
| فاصله‌ی اضافه یا حروف بزرگ و کوچک ناهمسان (قابل اصلاح) | ۷ | |
| سطر تکراری عیناً یکسان | ۴ | ۱۹، ۱۶۰، ۳۱۱، ۴۲۲ |
| نمره خارج از بازه (<code dir="ltr">145</code> به جای <code dir="ltr">14.5</code>، و <code dir="ltr">-3</code>) | ۲ | ۵۸، ۱۸۵ |
| اعشار با ویرگول (<code dir="ltr">17,25</code>) که سطر را شش‌بخشی می‌کند | ۱ | ۲۲۷ |
| سطر ناقص (بدون نام درس) | ۱ | ۲۷۶ |
| واحد ناسازگار با درس (<code dir="ltr">Calculus 1</code> با ۴ واحد) | ۱ | ۲ |
| دو نمره‌ی متفاوت برای یک دانشجو در یک درس | ۱ | ۴۰۴ |
| غیبت <code dir="ltr">ABS</code> در <code dir="ltr">Linear Algebra</code> (معتبر؛ نمره‌ی صفر) | ۳ | |

### الگوهای کاشته‌شده (الزام ۶)

- <code dir="ltr">Statistics 1</code>: درصد مردودی حدود ۴۱٪ در برابر ۲ تا ۱۷٪ در درس‌های دیگر، با میانگین حدود ۱۰٫۸.
- <code dir="ltr">Linear Algebra</code>: سه نمره‌ی صفر (غیبت)، که کمینه را صفر و انحراف معیار را بالا می‌برد. این درس از نظر آموزشی مشکلی ندارد؛ دانشجوی دقیق باید این را از درس دشوار تشخیص دهد.
- ورودی‌ها: میانگین معدل از ۱۴۰۰ به ۱۴۰۲ کم می‌شود و تعداد مشروطی‌ها بیشتر می‌شود.
- اثر پاک‌سازی روی رتبه‌ها: سطر ۲۲۷ (<code dir="ltr">17,25</code>) مربوط به نفر اول جدول (<code dir="ltr">Omid Bushehri</code>) است و سطر ۲۷۶ (ناقص) مربوط به <code dir="ltr">Neda Ahmadi</code> از ممتازها. دانشجویی که <code dir="ltr">17,25</code> را به <code dir="ltr">17.25</code> اصلاح کند، معدل نفر اول را کمی تغییر می‌دهد. این نمونه‌ی خوبی برای پرسش در دفاع است: «تصمیم پاک‌سازی شما روی کدام نتیجه اثر گذاشت؟»
- نتیجه‌ی پرسش سیاستی: ارفاق ۲ نمره‌ای به <code dir="ltr">Statistics 1</code> فقط یک نفر را از مشروطی خارج می‌کند؛ یعنی مشروطی‌ها عمدتاً به ضعف کلی دانشجو مربوط‌اند، نه به یک درس.

</div>

In [1]:
import math

import math

UNITS = {"Calculus 1": 3, "Physics 1": 3, "Programming": 3, "Statistics 1": 3,
         "Linear Algebra": 3, "General English": 2, "Discrete Math": 3, "Physical Education": 1}

def is_number(s):
    if s.startswith("-"):
        s = s[1:]
    parts = s.split(".")
    return 1 <= len(parts) <= 2 and all(p.isdigit() for p in parts)

def clean_data(raw_text):
    records, rejected = [], []
    seen = {}
    for line_no, line in enumerate(raw_text.splitlines(), start=1):
        if line.strip() == "" or line.startswith("student_id"):
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) != 5:
            rejected.append((line_no, line, "wrong number of fields")); continue
        sid, name, course, units, grade = parts
        name = name.title()
        # normalize course name against the official list
        match = None
        for c in UNITS:
            if c.lower() == course.lower():
                match = c
        if match is None:
            rejected.append((line_no, line, "unknown course")); continue
        course = match
        if not (sid.isdigit() and len(sid) == 7):
            rejected.append((line_no, line, "bad student id")); continue
        if not units.isdigit() or int(units) != UNITS[course]:
            rejected.append((line_no, line, "units do not match the course")); continue
        if grade.upper() == "ABS":
            g = 0.0
        elif is_number(grade) and 0 <= float(grade) <= 20:
            g = float(grade)
        else:
            rejected.append((line_no, line, "invalid grade")); continue
        key = (sid, course)
        rec = (sid, name, course, int(units), g)
        if key in seen:
            if seen[key] == rec:
                rejected.append((line_no, line, "exact duplicate")); continue
            rejected.append((line_no, line, "conflicting record")); continue
        seen[key] = rec
        records.append(rec)
    return records, rejected

def gpa(records, student_id):
    num = den = 0
    for sid, name, course, u, g in records:
        if sid == student_id:
            num += g * u; den += u
    return num / den if den else None

def median(xs):
    xs = sorted(xs); n = len(xs)
    return xs[n // 2] if n % 2 else (xs[n // 2 - 1] + xs[n // 2]) / 2

def std(xs):
    m = sum(xs) / len(xs)
    return math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))

In [2]:
RAW_DATA = """student_id,name,course,units,grade
4009414,Parisa Abbasi,Calculus 1,4,15.5
4014257,Mohammad Dashti,Linear Algebra,3,ABS
4018507,Mehdi Tangestani,Programming,3,16.75
 4006062 , OMID BUSHEHRI , physics 1 , 3 , 16.25 
4018850,Amir Najafi,Statistics 1,3,17.25
4016853,Babak Tangestani,Calculus 1,3,15.5
4004678,Yasaman Ahmadi,Linear Algebra,3,ABS
4007891,Hossein Ghasemi,Physics 1,3,13.5
4018420,Arash Rahimi,Calculus 1,3,8.5
4009414,Parisa Abbasi,General English,2,18.25
4029925,Amir Mohammadi,Discrete Math,3,16.5
4024088,Reza Salehi,Linear Algebra,3,ABS
4009975,Yasaman Bushehri,Programming,3,20
4027379,Ali Rezaei,Physical Education,1,14.75
4009045,Yasaman Dashti,Discrete Math,3,12.25
4028600,Amir Rahimi,Physics 1,3,17
4018420,Arash Rahimi,Programming,3,12.5
4029925,Amir Mohammadi,Discrete Math,3,16.5
4012583,Leila Bagheri,Physics 1,3,12.5
4007917,Parisa Salehi,Calculus 1,3,15.75
4027969,Hamid Rahimi,General English,2,17.5
4029513,Mehdi Bushehri,Physics 1,3,13
4023797,Leila Salehi,Statistics 1,3,12.5
4015261,Parisa Jafari,Discrete Math,3,7.5
4019955,Ali Salehi,Programming,3,15
4025713,Leila Najafi,Discrete Math,3,11.5
4025975,Leila Sadeghi,Calculus 1,3,10
4019060,Shirin Bushehri,Programming,3,16
4019145,Neda Shirazi,General English,2,17.75
4023623,Mohammad Najafi,General English,2,19.25
4008035,Elham Dashti,Programming,3,12.75
4009233,Hossein Sadeghi,Calculus 1,3,16
4009414,Parisa Abbasi,Linear Algebra,3,13.25
4006062,Omid Bushehri,General English,2,18
4012583,Leila Bagheri,Statistics 1,3,14.75
4022288,Kaveh Najafi,Statistics 1,3,7.5
4028600,Amir Rahimi,Physical Education,1,20
4003308,Fatemeh Salehi,Statistics 1,3,12.25
4024822,Neda Dashti,Linear Algebra,3,16.75
4018420,Arash Rahimi,General English,2,10.75
4029596,Hossein Shirazi,Calculus 1,3,14
4004444,babak sadeghi,  Statistics 1  ,3,8.25
4029513,Mehdi Bushehri,Physical Education,1,17.75
4019060,Shirin Bushehri,Physics 1,3,15.25
4009975,Yasaman Bushehri,General English,2,18.75
4026685,Mohammad Mohammadi,Linear Algebra,3,8
4024088,Reza Salehi,Statistics 1,3,3.75
4029596,Hossein Shirazi,Statistics 1,3,7.75
4018183,Neda Rezaei,Physical Education,1,17.75
4015261,Parisa Jafari,Physics 1,3,6.5
4028600,Amir Rahimi,Discrete Math,3,15
4015762,Mina Shirazi,General English,2,18.75
4008226,Leila Bushehri,Physical Education,1,14.25
4006062,Omid Bushehri,Discrete Math,3,19.5
4024822,Neda Dashti,Physics 1,3,15.25
4002680,Elham Shirazi,Discrete Math,3,11.25
4009233,Hossein Sadeghi,Discrete Math,3,145
4005237,Hamid Shirazi,Linear Algebra,3,17
4002322,Maryam Ahmadi,Linear Algebra,3,18
4015261,Parisa Jafari,Calculus 1,3,7.25

4008035,Elham Dashti,Physical Education,1,15.5
4016188,Reza Kazemi,General English,2,14
4011542,Zahra Ahmadi,Physics 1,3,12.5
4009045,Yasaman Dashti,Physics 1,3,11.25
4019060,Shirin Bushehri,Linear Algebra,3,14
4018183,Neda Rezaei,Calculus 1,3,12
4016188,Reza Kazemi,Physics 1,3,11.5
4029925,Amir Mohammadi,Physical Education,1,16.5
4009486,Leila Shirazi,Discrete Math,3,10
4023623,Mohammad Najafi,Statistics 1,3,15.25
4029967,Babak Hosseini,Calculus 1,3,12.25
4018507,Mehdi Tangestani,General English,2,17
4023623,Mohammad Najafi,Physics 1,3,19.25
4018183,Neda Rezaei,Discrete Math,3,12
4029513,Mehdi Bushehri,Statistics 1,3,11.75
4012583,Leila Bagheri,Calculus 1,3,14.5
4007891,Hossein Ghasemi,Linear Algebra,3,10.75
4018507,Mehdi Tangestani,Calculus 1,3,14.75
4025713,Leila Najafi,STATISTICS 1,3,8.5
4011542,Zahra Ahmadi,Programming,3,15.25
4002322,Maryam Ahmadi,Calculus 1,3,19
4015762,Mina Shirazi,Physical Education,1,19.75
4019060,Shirin Bushehri,Discrete Math,3,14.25
4019939,Babak Heydari,Programming,3,14.75
4029967,Babak Hosseini,Discrete Math,3,12.25
4009486,Leila Shirazi,Programming,3,11.75
4009975,Yasaman Bushehri,Discrete Math,3,18.25
4008484,Maryam Rezaei,Programming,3,15.75
4015399,Nima Tavakoli,Linear Algebra,3,17.25
4014257,Mohammad Dashti,General English,2,14.75
4022288,Kaveh Najafi,Physical Education,1,15.75
4007891,Hossein Ghasemi,Programming,3,15.5
4028600,Amir Rahimi,General English,2,16
4008226,Leila Bushehri,Discrete Math,3,11
4025713,Leila Najafi,Physical Education,1,13.75
4013397,Sara Najafi,Programming,3,16
4009166,Mohammad Abbasi,Discrete Math,3,17
4027379,Ali Rezaei,Calculus 1,3,13.75
4024088,Reza Salehi,Calculus 1,3,8.25
4019223,Fatemeh Tavakoli,Programming,3,16.75
4023797,Leila Salehi,Programming,3,17.5
4029318,Shirin Kazemi,Calculus 1,3,11.75
4027969,Hamid Rahimi,Statistics 1,3,16.75
4006577,Omid Hosseini,Calculus 1,3,18.25
4025975,Leila Sadeghi,Discrete Math,3,7.25
4016188,Reza Kazemi,Physical Education,1,14
4009032,Neda Ahmadi,Programming,3,17.5
4011542,Zahra Ahmadi,Statistics 1,3,10
4002680,Elham Shirazi,Statistics 1,3,9.5
4007917,Parisa Salehi,Physical Education,1,19.25
4007917,Parisa Salehi,Statistics 1,3,10.75
4024822,Neda Dashti,Discrete Math,3,16.25
4023256,Sara Shirazi,Linear Algebra,3,16
4029513,Mehdi Bushehri,Programming,3,14.5
4009414,Parisa Abbasi,Physics 1,3,15.75
4008035,Elham Dashti,Discrete Math,3,12.75
4029925,Amir Mohammadi,Physics 1,3,15.5
4029513,Mehdi Bushehri,Discrete Math,3,15.5
4007891,Hossein Ghasemi,Statistics 1,3,11
4018420,Arash Rahimi,Physics 1,3,10.75
4026685,Mohammad Mohammadi,Programming,3,7.5
 4016188 , REZA KAZEMI , programming , 3 , 9.75 
4023623,Mohammad Najafi,Linear Algebra,3,16
4009032,Neda Ahmadi,Calculus 1,3,19.75
4022288,Kaveh Najafi,Calculus 1,3,10.75
4013397,Sara Najafi,Linear Algebra,3,14
4029925,Amir Mohammadi,Calculus 1,3,15.5
4017585,Elham Heydari,Discrete Math,3,11.5
4027379,Ali Rezaei,Linear Algebra,3,14
4005237,Hamid Shirazi,General English,2,17.25
4018183,Neda Rezaei,Linear Algebra,3,12.25
4015261,Parisa Jafari,Physical Education,1,10
4004444,Babak Sadeghi,Physical Education,1,12.75
4025975,Leila Sadeghi,Programming,3,7.25
4009166,Mohammad Abbasi,Calculus 1,3,16.25
4027969,Hamid Rahimi,Physical Education,1,19.25
4025713,Leila Najafi,General English,2,12.25
4009486,Leila Shirazi,General English,2,11.5
4018507,Mehdi Tangestani,Linear Algebra,3,16.5
4006577,Omid Hosseini,General English,2,19.5
4019223,Fatemeh Tavakoli,Physics 1,3,15
4007891,Hossein Ghasemi,Discrete Math,3,14.5
4023433,Saeed Abbasi,Linear Algebra,3,10.75
4027379,Ali Rezaei,Programming,3,14
4003308,Fatemeh Salehi,General English,2,17.75
4019145,Neda Shirazi,Calculus 1,3,13
4009414,Parisa Abbasi,Statistics 1,3,12.25
4009032,Neda Ahmadi,Discrete Math,3,17
4009045,Yasaman Dashti,Physical Education,1,15.5
4023623,Mohammad Najafi,Physical Education,1,20
4004678,Yasaman Ahmadi,Discrete Math,3,15.75
4017585,Elham Heydari,Statistics 1,3,11.25
4023256,Sara Shirazi,Programming,3,17.5
4004678,Yasaman Ahmadi,Physics 1,3,13.5
4027969,Hamid Rahimi,Discrete Math,3,17.25
4024088,Reza Salehi,Discrete Math,3,8
4019145,Neda Shirazi,Linear Algebra,3,12.25
4004678,Yasaman Ahmadi,Discrete Math,3,15.75
4004678,Yasaman Ahmadi,General English,2,18
4029682,Ali Karimi,General English,2,13.75
4005237,Hamid Shirazi,Physical Education,1,20
4022288,Kaveh Najafi,General English,2,14.25
4029596,Hossein Shirazi,Discrete Math,3,13
4009233,Hossein Sadeghi,General English,2,16.25
4026685,Mohammad Mohammadi,Discrete Math,3,9
4009233,Hossein Sadeghi,Statistics 1,3,9
4018183,Neda Rezaei,General English,2,14.75
4025713,Leila Najafi,Linear Algebra,3,11.5
4004678,Yasaman Ahmadi,Calculus 1,3,15.25
4003308,Fatemeh Salehi,Calculus 1,3,16.5
4023623,Mohammad Najafi,Discrete Math,3,18
4023433,Saeed Abbasi,Programming,3,12.25
4029362,Ali Ahmadi,Physical Education,1,8.25
4008484,Maryam Rezaei,Physical Education,1,17.5
4012583,Leila Bagheri,General English,2,14.5
4019060,Shirin Bushehri,Statistics 1,3,7.75
4006577,Omid Hosseini,Discrete Math,3,19.5
4023797,Leila Salehi,Physics 1,3,18.5
4009032,Neda Ahmadi,Physical Education,1,19.25
4018420,Arash Rahimi,Physical Education,1,11.5
4015399,Nima Tavakoli,Statistics 1,3,14.5
4018183,Neda Rezaei,Programming,3,12.25
4029362,Ali Ahmadi,Calculus 1,3,-3
4017585,Elham Heydari,Physics 1,3,10.25
4029682,Ali Karimi,Programming,3,15
4002322,Maryam Ahmadi,Programming,3,14.5
4018850,Amir Najafi,Linear Algebra,3,16
4019942,Elham Karimi,Calculus 1,3,15
4009975,Yasaman Bushehri,Calculus 1,3,16
4025975,Leila Sadeghi,Physical Education,1,11.5
4003308,Fatemeh Salehi,Linear Algebra,3,14.5
4002680,Elham Shirazi,Calculus 1,3,8.75
4019145,Neda Shirazi,Physical Education,1,18.25
4008226,Leila Bushehri,Statistics 1,3,9.5
4027969,Hamid Rahimi,Calculus 1,3,16
4023797,Leila Salehi,Discrete Math,3,17
4019223,Fatemeh Tavakoli,Linear Algebra,3,15.5
4023433,Saeed Abbasi,General English,2,12.75
4019939,Babak Heydari,Physics 1,3,12.5
4025713,Leila Najafi,Physics 1,3,9.5
4003308,Fatemeh Salehi,Discrete Math,3,15
4017585,Elham Heydari,Physical Education,1,14.75
4018183,Neda Rezaei,STATISTICS 1,3,8
4029682,Ali Karimi,Physics 1,3,12.75
4009166,Mohammad Abbasi,Linear Algebra,3,14.75
4006062,Omid Bushehri,Programming,3,19.25
4029596,Hossein Shirazi,Programming,3,17
4007891,Hossein Ghasemi,Physical Education,1,18.25
4009975,Yasaman Bushehri,Physical Education,1,20
4029513,Mehdi Bushehri,Linear Algebra,3,12.75
4008226,Leila Bushehri,Programming,3,13
4029682,Ali Karimi,Statistics 1,3,8.5
4024088,Reza Salehi,General English,2,14
4029967,Babak Hosseini,Physical Education,1,14.25
4029318,Shirin Kazemi,Statistics 1,3,6.25
4023797,Leila Salehi,Physical Education,1,19.25
4029596,Hossein Shirazi,Physics 1,3,14
4004678,Yasaman Ahmadi,Statistics 1,3,9.75
4014257,Mohammad Dashti,Physics 1,3,15.75
4019223,Fatemeh Tavakoli,Physical Education,1,20
4002322,Maryam Ahmadi,General English,2,17.5
4019942,Elham Karimi,Linear Algebra,3,15.25
4015261,Parisa Jafari,Linear Algebra,3,10
4013397,Sara Najafi,Physics 1,3,13.25
4006062,Omid Bushehri,Statistics 1,3,17,25
4003308,Fatemeh Salehi,Programming,3,17.5
4009486,Leila Shirazi,Physics 1,3,11.25
4019942,Elham Karimi,Statistics 1,3,10.25
4028600,Amir Rahimi,Programming,3,12.5
4011542,Zahra Ahmadi,Calculus 1,3,18
4023256,Sara Shirazi,Statistics 1,3,13.5
4014257,Mohammad Dashti,Programming,3,13.25
4007917,Parisa Salehi,Programming,3,15.5
4026685,Mohammad Mohammadi,General English,2,11.75
4019060,Shirin Bushehri,General English,2,17.75
4019939,Babak Heydari,Physical Education,1,18.5
4009166,Mohammad Abbasi,Programming,3,18.5
4009233,Hossein Sadeghi,Physics 1,3,16.5
4023256,Sara Shirazi,Physical Education,1,19.25
4014257,Mohammad Dashti,Physical Education,1,18.25
4019955,Ali Salehi,General English,2,13.5
4019955,Ali Salehi,Statistics 1,3,9.75
4009166,Mohammad Abbasi,Physical Education,1,19.75
4008226,Leila Bushehri,Calculus 1,3,8
4027969,Hamid Rahimi,Physics 1,3,17
4008035,Elham Dashti,Physics 1,3,10.25
4019939,Babak Heydari,Discrete Math,3,15.75
4023256,Sara Shirazi,Calculus 1,3,14.75
4019223,Fatemeh Tavakoli,Statistics 1,3,11.75

4008484,Maryam Rezaei,Linear Algebra,3,16.5
4016188,Reza Kazemi,Discrete Math,3,12.5
4019223,Fatemeh Tavakoli,Calculus 1,3,17.5
4019955,Ali Salehi,Physical Education,1,14.75
4029362,Ali Ahmadi,Discrete Math,3,8
4019145,Neda Shirazi,Physics 1,3,15.25
4005237,Hamid Shirazi,Programming,3,15.75
4018850,Amir Najafi,Discrete Math,3,17.5
4016853,Babak Tangestani,Programming,3,19.5
4023623,Mohammad Najafi,Calculus 1,3,18
4023433,Saeed Abbasi,Discrete Math,3,11.5
4009166,Mohammad Abbasi,Physics 1,3,16.75
4018850,Amir Najafi,Calculus 1,3,15
4009486,Leila Shirazi,CALCULUS 1,3,13.25
4002322,Maryam Ahmadi,Discrete Math,3,16.5
4029318,Shirin Kazemi,Programming,3,15
4029967,Babak Hosseini,General English,2,12.75
4022288,Kaveh Najafi,Physics 1,3,12.5
4023433,Saeed Abbasi,Physical Education,1,15
4024088,Reza Salehi,Physics 1,3,8
4018420,Arash Rahimi,Discrete Math,3,13
4019223,Fatemeh Tavakoli,General English,2,16
4014257,Mohammad Dashti,Discrete Math,3,13.75
4009032,Neda Ahmadi,3,18.5
4009166,Mohammad Abbasi,Statistics 1,3,15.5
4027379,Ali Rezaei,Physics 1,3,14
4013397,Sara Najafi,General English,2,14
4002680,Elham Shirazi,Linear Algebra,3,7
4019942,Elham Karimi,Physical Education,1,17
4015762,Mina Shirazi,Statistics 1,3,13.25
4026685,Mohammad Mohammadi,Statistics 1,3,6.5
4005237,Hamid Shirazi,Calculus 1,3,17.5
4029682,Ali Karimi,Physical Education,1,17.75
4025975,Leila Sadeghi,Linear Algebra,3,9
4026685,Mohammad Mohammadi,Physics 1,3,12
4023433,Saeed Abbasi,Physics 1,3,12
4009045,Yasaman Dashti,Linear Algebra,3,11.75
4018850,Amir Najafi,Physical Education,1,20
4007891,Hossein Ghasemi,General English,2,13.75
4008035,Elham Dashti,Linear Algebra,3,9.75
4016853,Babak Tangestani,Statistics 1,3,11.5
4029318,Shirin Kazemi,General English,2,12.75
4004444,Babak Sadeghi,Calculus 1,3,13
4029362,Ali Ahmadi,General English,2,10
4018850,Amir Najafi,Programming,3,17.5
4013397,Sara Najafi,Physical Education,1,16.75
4002680,Elham Shirazi,Physics 1,3,9.75
4011542,Zahra Ahmadi,Linear Algebra,3,13
4029967,Babak Hosseini,Linear Algebra,3,16
4016853,Babak Tangestani,Physics 1,3,14.75
4029682,Ali Karimi,Linear Algebra,3,11.75
4016853,Babak Tangestani,General English,2,17.5
4025975,Leila Sadeghi,General English,2,9.75
4009414,Parisa Abbasi,Discrete Math,3,14.75
4015261,Parisa Jafari,Programming,3,9.25
4004678,Yasaman Ahmadi,Physical Education,1,19.25
4029362,Ali Ahmadi,Linear Algebra,3,6.5
4008484,Maryam Rezaei,General English,2,14.75
4016853,Babak Tangestani,General English,2,17.5
4018850,Amir Najafi,Physics 1,3,18
4004678,Yasaman Ahmadi,Programming,3,16.25
4024822,Neda Dashti,Physical Education,1,20
4006577,Omid Hosseini,Physics 1,3,18.25
4029925,Amir Mohammadi,Linear Algebra,3,12.75
4009045,Yasaman Dashti,Programming,3,14
4007917,Parisa Salehi,Discrete Math,3,9.75
4029967,Babak Hosseini,Statistics 1,3,11.75
4018850,Amir Najafi,General English,2,18.25
4009233,Hossein Sadeghi,Physical Education,1,20
4019060,Shirin Bushehri,Physical Education,1,17.5
4008226,Leila Bushehri,Linear Algebra,3,8.75
4024822,Neda Dashti,Statistics 1,3,13
4023797,Leila Salehi,Linear Algebra,3,18.25
4006577,Omid Hosseini,Programming,3,14.5
4002680,Elham Shirazi,General English,2,12.25
4008484,Maryam Rezaei,Physics 1,3,11.75
4009032,Neda Ahmadi,Linear Algebra,3,18.25
4019145,Neda Shirazi,Statistics 1,3,11.75
4026685,Mohammad Mohammadi,Physical Education,1,13.25
4013397,Sara Najafi,Statistics 1,3,13
4024822,Neda Dashti,Calculus 1,3,14.25
4025975,Leila Sadeghi,Physics 1,3,7
4023797,Leila Salehi,Calculus 1,3,19.75
4029925,Amir Mohammadi,Programming,3,14.25
4002322,Maryam Ahmadi,Physics 1,3,18.25
4017585,Elham Heydari,Programming,3,12.25
4008035,Elham Dashti,Calculus 1,3,15.5
 4019939 , BABAK HEYDARI , general english , 2 , 15.25 
4018507,Mehdi Tangestani,Physical Education,1,17.75
4029596,Hossein Shirazi,Linear Algebra,3,15
4006577,Omid Hosseini,Physical Education,1,20
4004444,Babak Sadeghi,General English,2,11.5
4019942,Elham Karimi,Physics 1,3,15
4019955,Ali Salehi,Physics 1,3,14
4022288,Kaveh Najafi,Programming,3,14
4018420,Arash Rahimi,Linear Algebra,3,10
4013397,Sara Najafi,Calculus 1,3,13.25
4006577,Omid Hosseini,Statistics 1,3,14
4023433,Saeed Abbasi,Calculus 1,3,12.25
4003308,Fatemeh Salehi,Physical Education,1,18.25
4029513,Mehdi Bushehri,General English,2,14.5
4029682,Ali Karimi,Discrete Math,3,14.25
4007917,Parisa Salehi,General English,2,17
4005237,Hamid Shirazi,Physics 1,3,17
4016853,Babak Tangestani,Linear Algebra,3,15
4007917,Parisa Salehi,Linear Algebra,3,13.5
4009045,Yasaman Dashti,General English,2,12.25
4009486,Leila Shirazi,Linear Algebra,3,12.75
4013397,Sara Najafi,Discrete Math,3,16
4029318,Shirin Kazemi,Physical Education,1,11.5
4024088,Reza Salehi,Physical Education,1,16
4011542,Zahra Ahmadi,General English,2,17.5
4011542,Zahra Ahmadi,Discrete Math,3,13.75
4017585,Elham Heydari,Calculus 1,3,11.75
4014257,Mohammad Dashti,Calculus 1,3,13.25
4016853,Babak Tangestani,Physical Education,1,17.75
4002680,Elham Shirazi,Physical Education,1,13.5
4018507,Mehdi Tangestani,Statistics 1,3,10.5
4004444,Babak Sadeghi,Physics 1,3,12.5
4016853,Babak Tangestani,Discrete Math,3,13.25
4008035,Elham Dashti,Statistics 1,3,9
4029513,Mehdi Bushehri,Calculus 1,3,16.5
4012583,Leila Bagheri,Discrete Math,3,13.5
4008484,Maryam Rezaei,Statistics 1,3,9.5
4015399,Nima Tavakoli,Physics 1,3,17.25
4029362,Ali Ahmadi,Programming,3,9
4023623,Mohammad Najafi,Programming,3,18.25
4029362,Ali Ahmadi,Physics 1,3,9
4009032,Neda Ahmadi,Statistics 1,3,12.5
4009486,Leila Shirazi,Physical Education,1,16.5
4009045,Yasaman Dashti,Calculus 1,3,10.25
4014257,Mohammad Dashti,Statistics 1,3,8.5
4006062,Omid Bushehri,Physical Education,1,19.5
4004444,Babak Sadeghi,Linear Algebra,3,12.5
4028600,Amir Rahimi,Calculus 1,3,12
4015399,Nima Tavakoli,Physical Education,1,20
4028600,Amir Rahimi,Statistics 1,3,8.5
4006062,Omid Bushehri,Linear Algebra,3,17
4015762,Mina Shirazi,Discrete Math,3,17.75
4027379,Ali Rezaei,General English,2,16.25
4009975,Yasaman Bushehri,Physics 1,3,17.5
4002322,Maryam Ahmadi,Statistics 1,3,11.25
4015399,Nima Tavakoli,Programming,3,19.75
4023256,Sara Shirazi,Discrete Math,3,14.25
4019942,Elham Karimi,General English,2,17.25
4019145,Neda Shirazi,Programming,3,16
4025713,Leila Najafi,Calculus 1,3,11.5
4019223,Fatemeh Tavakoli,Discrete Math,3,16.5
4029318,Shirin Kazemi,Linear Algebra,3,11.75
4029596,Hossein Shirazi,Physical Education,1,15
4015762,Mina Shirazi,Physics 1,3,17.5
4009166,Mohammad Abbasi,Discrete Math,3,20
4006062,Omid Bushehri,Calculus 1,3,19
4012583,Leila Bagheri,Linear Algebra,3,13
4029967,Babak Hosseini,Programming,3,14
4005237,Hamid Shirazi,Discrete Math,3,14.5
4019939,Babak Heydari,Calculus 1,3,13.25
4016188,Reza Kazemi,Statistics 1,3,5.5
4015762,Mina Shirazi,Calculus 1,3,18
4009233,Hossein Sadeghi,Linear Algebra,3,16.5
4004444,Babak Sadeghi,Programming,3,15
4027379,Ali Rezaei,Discrete Math,3,12
4023256,Sara Shirazi,General English,2,17
4019955,Ali Salehi,Calculus 1,3,14.25
4019939,Babak Heydari,Statistics 1,3,10.75
4019955,Ali Salehi,Discrete Math,3,13
4029925,Amir Mohammadi,Statistics 1,3,12
4024822,Neda Dashti,Programming,3,19.75
4008035,Elham Dashti,General English,2,14.25
4023256,Sara Shirazi,General English,2,17
4018507,Mehdi Tangestani,Discrete Math,3,16.5
4016188,Reza Kazemi,Linear Algebra,3,10.25
4009414,Parisa Abbasi,Physical Education,1,20
4015762,Mina Shirazi,Linear Algebra,3,17.25
4015399,Nima Tavakoli,Calculus 1,3,17

"""

<div dir="rtl">

## الزام ۱: پاک‌سازی

</div>

In [3]:
records, rejected = clean_data(RAW_DATA)
print(len(records), "clean records;", len(rejected), "rejected")
for r in rejected:
    print(r)

414 clean records; 10 rejected
(2, '4009414,Parisa Abbasi,Calculus 1,4,15.5', 'units do not match the course')
(19, '4029925,Amir Mohammadi,Discrete Math,3,16.5', 'exact duplicate')
(58, '4009233,Hossein Sadeghi,Discrete Math,3,145', 'invalid grade')
(160, '4004678,Yasaman Ahmadi,Discrete Math,3,15.75', 'exact duplicate')
(185, '4029362,Ali Ahmadi,Calculus 1,3,-3', 'invalid grade')
(227, '4006062,Omid Bushehri,Statistics 1,3,17,25', 'wrong number of fields')
(276, '4009032,Neda Ahmadi,3,18.5', 'wrong number of fields')
(311, '4016853,Babak Tangestani,General English,2,17.5', 'exact duplicate')
(404, '4009166,Mohammad Abbasi,Discrete Math,3,20', 'conflicting record')
(422, '4023256,Sara Shirazi,General English,2,17', 'exact duplicate')


<div dir="rtl">

## الزامات ۲، ۳ و ۸: وضعیت، رتبه و فهرست‌ها

</div>

In [4]:
names = {}
for sid, name, c, u, g in records:
    names[sid] = name

def status(g):
    return "probation" if g < 12 else ("honors" if g >= 17 else "normal")

def ranking(records):
    table = []
    for sid in names:
        table.append((round(gpa(records, sid), 2), sid))
    table.sort(reverse=True)
    ranked = []
    for i, (g, sid) in enumerate(table):
        rank = i + 1 if i == 0 or g != table[i - 1][0] else ranked[-1][0]
        ranked.append((rank, sid, names[sid], g))
    return ranked

ranked = ranking(records)
for row in ranked[:10]:
    print(row)

units_taken = {}
units_passed = {}
for sid, name, c, u, g in records:
    units_taken[sid] = units_taken.get(sid, 0) + u
    if g >= 10:
        units_passed[sid] = units_passed.get(sid, 0) + u

counts = {}
for rank, sid, name, g in ranked:
    counts[status(g)] = counts.get(status(g), 0) + 1
print(counts)
print("probation:", [(sid, name, g) for rank, sid, name, g in reversed(ranked) if g < 12])
print("honors:   ", [(sid, name, g) for rank, sid, name, g in ranked if g >= 17])
print("example:", ranked[0][1], units_taken[ranked[0][1]], units_passed.get(ranked[0][1], 0))

(1, '4006062', 'Omid Bushehri', 18.25)
(2, '4009975', 'Yasaman Bushehri', 18.18)
(3, '4023623', 'Mohammad Najafi', 17.75)
(4, '4023797', 'Leila Salehi', 17.36)
(4, '4006577', 'Omid Hosseini', 17.36)
(6, '4015399', 'Nima Tavakoli', 17.33)
(7, '4018850', 'Amir Najafi', 17.15)
(8, '4015762', 'Mina Shirazi', 17.14)
(8, '4009032', 'Neda Ahmadi', 17.14)
(10, '4027969', 'Hamid Rahimi', 17.02)
{'honors': 10, 'normal': 36, 'probation': 14}
probation: [('4024088', 'Reza Salehi', 7.11), ('4015261', 'Parisa Jafari', 8.22), ('4029362', 'Ali Ahmadi', 8.38), ('4025975', 'Leila Sadeghi', 8.47), ('4026685', 'Mohammad Mohammadi', 9.21), ('4002680', 'Elham Shirazi', 9.82), ('4008226', 'Leila Bushehri', 10.31), ('4016188', 'Reza Kazemi', 10.58), ('4025713', 'Leila Najafi', 10.88), ('4018420', 'Arash Rahimi', 10.96), ('4029318', 'Shirin Kazemi', 11.42), ('4014257', 'Mohammad Dashti', 11.49), ('4017585', 'Elham Heydari', 11.61), ('4022288', 'Kaveh Najafi', 11.9)]
honors:    [('4006062', 'Omid Bushehri', 18.

<div dir="rtl">

## الزام ۴: آمار درس‌ها و نمودار متنی

</div>

In [5]:
BINS = [(0, 10), (10, 12), (12, 14), (14, 16), (16, 18), (18, 20.01)]
for c in UNITS:
    gs = [g for s, n, cc, u, g in records if cc == c]
    fail = sum(1 for g in gs if g < 10) / len(gs)
    print(f"{c:<19} n={len(gs):3} mean={sum(gs)/len(gs):5.2f} median={median(gs):5.2f} "
          f"sd={std(gs):4.2f} min={min(gs):5.2f} max={max(gs):5.2f} fail={fail:5.1%}")
    for lo, hi in BINS:
        k = sum(1 for g in gs if lo <= g < hi)
        print(f"    [{lo:>2},{min(hi, 20):>2}) {'#' * k}")

Calculus 1          n= 52 mean=14.25 median=14.62 sd=3.12 min= 7.25 max=19.75 fail= 9.6%
    [ 0,10) #####
    [10,12) ######
    [12,14) ###########
    [14,16) #############
    [16,18) #########
    [18,20) ########
Physics 1           n= 51 mean=13.80 median=14.00 sd=3.09 min= 6.50 max=19.25 fail=11.8%
    [ 0,10) ######
    [10,12) #######
    [12,14) ############
    [14,16) ############
    [16,18) #########
    [18,20) #####
Programming         n= 51 mean=14.88 median=15.00 sd=2.98 min= 7.25 max=20.00 fail= 9.8%
    [ 0,10) #####
    [10,12) #
    [12,14) ########
    [14,16) ##################
    [16,18) ############
    [18,20) #######
Statistics 1        n= 49 mean=10.77 median=10.75 sd=2.84 min= 3.75 max=17.25 fail=40.8%
    [ 0,10) ####################
    [10,12) #############
    [12,14) #########
    [14,16) #####
    [16,18) ##
    [18,20) 
Linear Algebra      n= 52 mean=12.71 median=13.12 sd=4.29 min= 0.00 max=18.25 fail=17.3%
    [ 0,10) #########
    [10,12) ######

<div dir="rtl">

## الزام ۵: مقایسه‌ی ورودی‌ها

</div>

In [6]:
for year in ["400", "401", "402"]:
    gs = [g for rank, sid, name, g in ranked if sid.startswith(year)]
    n_prob = sum(1 for g in gs if g < 12)
    print(year, len(gs), round(sum(gs) / len(gs), 2), n_prob, f"{n_prob / len(gs):.0%}")

400 20 14.43 2 10%
401 20 13.77 5 25%
402 20 12.97 7 35%


<div dir="rtl">

## الزام ۷: پرسش سیاستی

</div>

In [7]:
worst = max(UNITS, key=lambda c: sum(1 for r in records if r[2] == c and r[4] < 10)
            / sum(1 for r in records if r[2] == c))
curved = [(s, n, c, u, min(20, g + 2) if c == worst else g) for s, n, c, u, g in records]
ranked2 = ranking(curved)
before = {sid for rank, sid, name, g in ranked if g < 12}
after = {sid for rank, sid, name, g in ranked2 if g < 12}
gs = [r[4] for r in curved if r[2] == worst]
print("course:", worst)
print("probation before/after:", len(before), len(after), "left probation:", before - after)
print("fail rate after:", f"{sum(1 for g in gs if g < 10) / len(gs):.1%}")
for row in ranked2[:10]:
    print(row)

course: Statistics 1
probation before/after: 14 13 left probation: {'4022288'}
fail rate after: 14.3%
(1, '4006062', 'Omid Bushehri', 18.25)
(2, '4009975', 'Yasaman Bushehri', 18.18)
(3, '4023623', 'Mohammad Najafi', 18.04)
(4, '4015399', 'Nima Tavakoli', 17.7)
(5, '4006577', 'Omid Hosseini', 17.69)
(6, '4023797', 'Leila Salehi', 17.67)
(7, '4009032', 'Neda Ahmadi', 17.52)
(8, '4015762', 'Mina Shirazi', 17.47)
(9, '4018850', 'Amir Najafi', 17.44)
(10, '4027969', 'Hamid Rahimi', 17.42)
